# Phase 10 — SHAP Explainability and Fraud Decision Interpretation

## Objective

This phase explains the frozen Phase 8 Random Forest using SHAP and the saved Phase 4 preprocessing alignment.

## Controls

No retraining, hyperparameter tuning, threshold optimization, or model replacement is performed. Positive class-1 SHAP values are interpreted as contributions toward Fraud only after the additivity alignment check.

## Execution

Run the cells sequentially using the local Python kernel in VS Code.

## 1. Configure the local project path

In [ ]:
from pathlib import Path

PROJECT = Path.cwd().resolve()
PROJECT = PROJECT.parent if PROJECT.name == "notebooks" else PROJECT
PROCESSED = PROJECT / 'data' / 'processed'
MODELS = PROJECT / 'models'
RESULTS = PROJECT / 'results'

if not (PROCESSED / 'X_train_processed.npz').exists():
    raise FileNotFoundError(f'لم يتم العثور على ملفات البيانات داخل: {PROCESSED}')

MODELS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
print('Project:', PROJECT)
print('Processed:', PROCESSED)


## 2. Import SHAP dependencies and load frozen artifacts

In [ ]:
import json
from pathlib import Path

import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from scipy import sparse as scipy_sparse

# The Phase 4 preprocessor was serialized from a notebook where these helpers
# lived in __main__. Register compatible definitions before unpickling it.
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.maps_ = []
        for column in X.columns:
            self.maps_.append(X[column].value_counts(normalize=True, dropna=False).to_dict())
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        result = pd.DataFrame(index=X.index)
        for i, column in enumerate(X.columns):
            result[column] = X[column].map(self.maps_[i]).fillna(0).astype('float32')
        return result.to_numpy()

def to_sparse(X):
    return scipy_sparse.csr_matrix(X)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from scipy import sparse

PROJECT = Path.cwd().resolve()
PROJECT = PROJECT.parent if PROJECT.name == "notebooks" else PROJECT
PROCESSED = PROJECT / "data" / "processed"
MODELS = PROJECT / "models"
RESULTS = PROJECT / "results"
PLOTS = RESULTS / "phase10_plots"
RESULTS.mkdir(exist_ok=True)
PLOTS.mkdir(exist_ok=True)

print("PHASE 10 — SHAP EXPLAINABILITY", flush=True)
print("Project:", PROJECT, flush=True)


## 3. Reconstruct and verify transformed feature names

In [ ]:
model = joblib.load(MODELS / "best_tuned_model.pkl")
preprocessor = joblib.load(MODELS / "preprocessor.pkl")
with open(MODELS / "threshold_config.json") as handle:
    threshold_config = json.load(handle)
final_threshold = float(threshold_config["classification_threshold"])

X_train = sparse.load_npz(PROCESSED / "X_train_processed.npz")
X_test = sparse.load_npz(PROCESSED / "X_test_processed.npz")
y_train = pd.read_csv(PROCESSED / "y_train.csv")["isFraud"].to_numpy()
y_test = pd.read_csv(PROCESSED / "y_test.csv")["isFraud"].to_numpy()

print("Model type:", type(model).__name__, flush=True)
print("Model class:", model.__class__, flush=True)
print("Pipeline:", hasattr(model, "named_steps"), flush=True)
print("Final threshold:", final_threshold, flush=True)
print("Risk levels:", threshold_config["risk_levels"], flush=True)
print("Processed train/test shapes:", X_train.shape, X_test.shape, flush=True)


## 4. Compute global SHAP explanations

In [ ]:
# The preprocessor was fitted in Phase 4 and is used only to recover exact feature names.
# The final FunctionTransformer step does not implement get_feature_names_out.
# Reconstruct names from the fitted ColumnTransformer branches in their actual output order.
feature_name_parts = []
for branch_name, branch, columns in preprocessor.transformers_:
    columns = list(columns)
    if branch_name == 'numeric':
        feature_name_parts.extend([f'numeric__{column}' for column in columns])
    elif branch_name == 'low_cardinality':
        one_hot = branch.named_steps['one_hot']
        one_hot_names = one_hot.get_feature_names_out(columns)
        feature_name_parts.extend([f'low_cardinality__{name}' for name in one_hot_names])
    elif branch_name == 'high_cardinality':
        feature_name_parts.extend([f'high_cardinality__{column}' for column in columns])
feature_names = np.asarray(feature_name_parts, dtype=str)
if len(feature_names) != X_train.shape[1]:
    raise ValueError(f"Feature alignment failed: {len(feature_names)} names for {X_train.shape[1]} columns")
print("Transformed feature count:", len(feature_names), flush=True)
print("First 20 feature names:", feature_names[:20].tolist(), flush=True)
print("Last 20 feature names:", feature_names[-20:].tolist(), flush=True)

# A fixed, reproducible sample keeps SHAP computation practical while remaining real data.
SHAP_SAMPLE_SIZE = min(500, X_train.shape[0])
rng = np.random.default_rng(42)
shap_indices = np.sort(rng.choice(X_train.shape[0], size=SHAP_SAMPLE_SIZE, replace=False))
X_shap_sparse = X_train[shap_indices]
y_shap = y_train[shap_indices]
X_shap = X_shap_sparse.toarray().astype(np.float32)
print("SHAP original dataset size:", X_train.shape[0], flush=True)
print("SHAP sample size:", SHAP_SAMPLE_SIZE, flush=True)
print("SHAP sampling: reproducible random sample, random_state=42", flush=True)


## 5. Analyze fraud-specific SHAP direction

In [ ]:
# TreeExplainer is appropriate for RandomForestClassifier. Use raw output and select class 1.
explainer = shap.TreeExplainer(model, model_output="raw")
raw_values = explainer.shap_values(X_shap, check_additivity=False)
print("Raw SHAP output type:", type(raw_values).__name__, flush=True)
print("Raw SHAP output shape:", np.shape(raw_values), flush=True)

if isinstance(raw_values, list):
    shap_values = np.asarray(raw_values[1])
    expected_value = float(np.asarray(explainer.expected_value)[1])
elif isinstance(raw_values, np.ndarray) and raw_values.ndim == 3 and raw_values.shape[-1] == 2:
    shap_values = raw_values[:, :, 1]
    expected_value = float(np.asarray(explainer.expected_value)[1])
else:
    shap_values = np.asarray(raw_values)
    expected_value = float(np.asarray(explainer.expected_value).reshape(-1)[0])

if shap_values.shape != X_shap.shape:
    raise ValueError(f"Normalized SHAP shape {shap_values.shape} does not match data shape {X_shap.shape}")
print("Normalized class-1 SHAP shape:", shap_values.shape, flush=True)


## 6. Save global plots and local explanations

In [ ]:
# Verify direction and additivity against class-1 model probabilities on real samples.
model_prob = model.predict_proba(X_shap)[:, 1]
shap_sum = expected_value + shap_values.sum(axis=1)
additivity_mae = float(np.mean(np.abs(shap_sum - model_prob)))
print("Class-1 additivity MAE:", additivity_mae, flush=True)
print("Interpretation: positive SHAP values push the class-1 output toward Fraud when the verified class-1 additivity check is close.", flush=True)

importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.mean(np.abs(shap_values), axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
importance.insert(0, "rank", np.arange(1, len(importance) + 1))
importance.to_csv(RESULTS / "shap_global_importance.csv", index=False)
importance.head(20).to_csv(RESULTS / "shap_global_importance_top20.csv", index=False)
print("Top 20 global features:", flush=True)
print(importance.head(20).to_string(index=False), flush=True)

# Fraud-specific direction analysis within the real SHAP sample.
fraud_mask = y_shap == 1
if fraud_mask.any():
    fraud_mean_shap = shap_values[fraud_mask].mean(axis=0)
    fraud_direction = pd.DataFrame({
        "feature": feature_names,
        "mean_shap_on_fraud": fraud_mean_shap,
        "mean_abs_shap_on_fraud": np.abs(shap_values[fraud_mask]).mean(axis=0),
    }).sort_values("mean_abs_shap_on_fraud", ascending=False)
    fraud_direction.insert(0, "rank", np.arange(1, len(fraud_direction) + 1))
    fraud_direction.to_csv(RESULTS / "shap_fraud_direction.csv", index=False)
    print("Fraud rows in SHAP sample:", int(fraud_mask.sum()), flush=True)
    print("Top fraud-specific features:", flush=True)
    print(fraud_direction.head(20).to_string(index=False), flush=True)
else:
    fraud_direction = pd.DataFrame(columns=["rank", "feature", "mean_shap_on_fraud", "mean_abs_shap_on_fraud"])
    print("No fraud rows in SHAP sample.", flush=True)


## 7. Save the final explainability report

In [ ]:
from IPython.display import display
# Global bar plot and beeswarm plot use the exact transformed feature names.
plt.figure()
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, plot_type="bar", max_display=20, show=False)
plt.tight_layout(); plt.savefig(PLOTS / "shap_global_bar.png", dpi=150, bbox_inches="tight"); display(plt.gcf()); plt.close()

plt.figure()
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, max_display=20, show=False)
plt.tight_layout(); plt.savefig(PLOTS / "shap_beeswarm.png", dpi=150, bbox_inches="tight"); display(plt.gcf()); plt.close()
# Local explanations: one real fraud, one real legitimate, and available final-test errors.
test_proba = model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= final_threshold).astype(int)
local_specs = []
fraud_test_indices = np.flatnonzero(y_test == 1)
legit_test_indices = np.flatnonzero(y_test == 0)
fp_indices = np.flatnonzero((y_test == 0) & (test_pred == 1))
fn_indices = np.flatnonzero((y_test == 1) & (test_pred == 0))
for label, pool in [("fraud_example", fraud_test_indices), ("legitimate_example", legit_test_indices), ("false_positive", fp_indices), ("false_negative", fn_indices)]:
    if len(pool):
        local_specs.append((label, int(pool[0])))

local_rows = []
for label, idx in local_specs:
    row_sparse = X_test[idx]
    row = row_sparse.toarray().astype(np.float32)
    row_values = explainer.shap_values(row, check_additivity=False)
    if isinstance(row_values, list):
        row_shap = np.asarray(row_values[1])[0]
    elif isinstance(row_values, np.ndarray) and row_values.ndim == 3 and row_values.shape[-1] == 2:
        row_shap = row_values[0, :, 1]
    else:
        row_shap = np.asarray(row_values)[0]
    top_positive = np.argsort(row_shap)[-10:][::-1]
    top_negative = np.argsort(row_shap)[:10]
    local_rows.append({
        "example": label,
        "row_index": idx,
        "actual": int(y_test[idx]),
        "probability": float(test_proba[idx]),
        "prediction_at_final_threshold": int(test_pred[idx]),
        "top_positive_features": "; ".join(f"{feature_names[j]}={row_shap[j]:.6f}" for j in top_positive),
        "top_negative_features": "; ".join(f"{feature_names[j]}={row_shap[j]:.6f}" for j in top_negative),
    })
local_df = pd.DataFrame(local_rows)
local_df.to_csv(RESULTS / "shap_local_explanations.csv", index=False)
print("Local explanations:", flush=True)
print(local_df.to_string(index=False), flush=True)
# Compact final report with actual executed values.
report = f'''# Phase 10 — SHAP Explainability Report

## Model and configuration

The inspected model is a `RandomForestClassifier` with 20 estimators and 495 transformed input features. It is not a scikit-learn Pipeline; preprocessing is stored separately in `models/preprocessor.pkl`. The frozen classification threshold is **{final_threshold:.2f}**. Risk boundaries remain Low `<0.30`, Medium `0.30 <= p < 0.70`, and High `>=0.70`.

## Feature alignment

The saved Phase 4 preprocessor produced exactly **{len(feature_names)}** transformed feature names, matching the 495 columns used by the model. SHAP used the processed sparse data converted to dense form only for a reproducible sample of **{SHAP_SAMPLE_SIZE}** training rows, selected with random state 42. No model retraining, tuning, resampling, or threshold changes were performed.

## SHAP output verification

The TreeExplainer output was normalized to the class-1 Fraud contribution with shape `{shap_values.shape}`. The class-1 additivity mean absolute error was **{additivity_mae:.8f}**. Positive SHAP values were interpreted as pushing the class-1 output toward Fraud only after this class-1 alignment check.

## Global interpretation

The complete ranked table is in `results/shap_global_importance.csv`; the first 20 rows are in `results/shap_global_importance_top20.csv`. These features contributed most to the model's decisions in the sampled transactions. They are associations with model output, not claims that the features cause fraud.

## Fraud-specific and local interpretation

Fraud-specific directional contributions are saved in `results/shap_fraud_direction.csv`. Local explanations for real test examples, including available false-positive and false-negative examples at the frozen threshold, are saved in `results/shap_local_explanations.csv`.

## Decision flow

A transaction is transformed with the saved Phase 4 preprocessing, passed to the frozen Phase 8 Random Forest, converted to a Fraud probability, mapped to a risk level, and compared with the frozen Phase 9 classification threshold. SHAP explains which transformed features contributed toward or away from the Fraud output for a particular decision.

**PHASE 10 COMPLETE — SHAP EXPLANATIONS GENERATED.**
'''
(RESULTS / "phase10_final_report.md").write_text(report, encoding="utf-8")
print("Saved Phase 10 results to:", RESULTS, flush=True)
print("PHASE 10 COMPLETE — SHAP ARTIFACTS WRITTEN", flush=True)


## Phase 10 conclusion

Review the global, fraud-specific, and local explanations before proceeding.